In [9]:
pip install tensorflow keras scikit-learn

## Data Processing

In [10]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [13]:
# Step 1 : Load the Data

def preprocess_selected_params(csv_path, seq_len=24) :
  df = pd.read_csv(csv_path)

  # Simulate CO if not present
  if 'co' not in df.columns :
    np.random.seed(42)
    df['co'] = np.random.normal(loc=0.8, scale=0.3, size=len(df))

  # Drop rows with missing target or required inputs :
  df = df.dropna(subset=["pm2.5", "TEMP", "DEWP", "co"])

  # keep only selected features
  df = df[["co", "TEMP", "DEWP", "pm2.5"]]

  # Normalize
  scaler = MinMaxScaler()
  scaled_data = scaler.fit_transform(df)

  joblib.dump(scaler, 'scaler.pkl')

  # Create time series sequences

  X, y = [], []
  for i in range(len(scaled_data) - seq_len) :
    X.append(scaled_data[i:i+seq_len])
    y.append(scaled_data[i+seq_len,3])  #pm2.5 is at index 3

  return np.array(X), np.array(y)

## Model Parameters

In [14]:
# Step 3 : Process Data
SEQ_LEN= 24

X, y = preprocess_selected_params('PRSA_data_2010.1.1-2014.12.31.csv',seq_len=SEQ_LEN)

# Step 4 : Train Test Split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Step 5 : Define and Train LSTM
model = Sequential()
model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Train Model

In [17]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1,
          callbacks=[EarlyStopping(patience=3)])

Epoch 1/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 5.7831e-04 - val_loss: 4.4709e-04
Epoch 2/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 5.8216e-04 - val_loss: 4.4899e-04
Epoch 3/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 5.7935e-04 - val_loss: 4.4120e-04
Epoch 4/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 5.7539e-04 - val_loss: 4.4562e-04
Epoch 5/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 5.7759e-04 - val_loss: 4.6036e-04
Epoch 6/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 5.7669e-04 - val_loss: 4.4385e-04


In [18]:
model.save("lstm_model.keras")

In [19]:
# Load model
model = load_model("lstm_model.keras")

# Predict again
future_prediction = model.predict(X_test)

261/261 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [20]:
future_prediction

array([[0.06159829],
       [0.09349583],
       [0.18211694],
       ...,
       [0.0154012 ],
       [0.01447721],
       [0.01103131]], dtype=float32)